# Study 827 — Cross-Asset Skewness Premium — the teardown

The per-leg books, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation asset-label placebo, the two-era and multi-window robustness cut, the costed monthly timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2007-01-03', 'end': '2026-06-30', 'n_classes': 9, 'n_months': 227, 'median_n': 9, 'fingerprint': '9ce7d7c0e243', 'spread_bps': 13.73, 't_nw': 0.62, 't_1s': 0.64, 'lo_bps': 63.24, 'hi_bps': 49.51, 'welch_t': 0.39, 'sharpe': 0.15, 'placebo_obs': 13.73, 'placebo_mean': -0.421, 'placebo_sd': 17.101, 'placebo_p': 0.204, 'placebo_sigma': 0.83, 'placebo_draws': 1000, 'era_early_bps': -15.21, 'era_early_t': -0.45, 'era_early_n': 107, 'era_late_bps': 39.54, 'era_late_t': 1.39, 'era_late_n': 120, 'w63_bps': 22.06, 'w63_t': 1.04, 'w252_bps': 5.95, 'w252_t': 0.25, 'timer_1_gross': 13.73, 'timer_1_cost': 6.17, 'timer_1_net': 7.57, 'timer_1_t': 0.35, 'timer_1_ann': 0.9, 'timer_5_gross': 13.73, 'timer_5_cost': 14.17, 'timer_5_net': -0.43, 'timer_5_t': -0.02, 'timer_5_ann': -0.1, 'null_mean_t': 0.01, 'null_sd_t': 1.22, 'null_fire': 2, 'planted_t': 2.52, 'planted_welch': 2.87}

## The headline — long-low-skew / short-high-skew spread

Monthly equal-weight bottom-⅓ minus top-⅓ realized-skew spread across nine classes (n = 227 months).

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/month  NW(6) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : low-skew {R['lo_bps']:+.2f} vs high-skew {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['sharpe']:.2f} (before cost, annualised)")

spread        : +13.73 bps/month  NW(6) t = +0.62  one-sample t = +0.64
books         : low-skew +63.24 vs high-skew +49.51 bps (Welch t = +0.39)
gross Sharpe  : 0.15 (before cost, annualised)


## Placebo — asset-label-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.4f}  (~{R['placebo_sigma']:+.2f} sigma)")

observed +13.73 bps vs placebo mean -0.421 (sd 17.101) -> p = 0.2040  (~+0.83 sigma)


## Robustness — two eras (split 2016-07-01) and the window sweep

In [4]:
print(f"2007-2016 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2016-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print(f" 63d window : {R['w63_bps']:+.2f} bps  NW t = {R['w63_t']:+.2f}")
print(f"252d window : {R['w252_bps']:+.2f} bps  NW t = {R['w252_t']:+.2f}")
print('  -> sign flips across eras; never clears |t|=2 at any lookback')

2007-2016 (n=107): -15.21 bps  NW t = -0.45
2016-2026 (n=120): +39.54 bps  NW t = +1.39
 63d window : +22.06 bps  NW t = +1.04
252d window : +5.95 bps  NW t = +0.25
  -> sign flips across eras; never clears |t|=2 at any lookback


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per monthly rebalance; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/mo (cost {c:.2f}/mo, t={t:+.2f})")

 1 bp one-way: gross +13.73 -> net +7.57 bps/mo (cost 6.17/mo, t=+0.35)
5 bps one-way: gross +13.73 -> net -0.43 bps/mo (cost 14.17/mo, t=-0.02)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation even on a nine-asset cross-section.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cross_asset_skew import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=827+s, n_assets=9, n_days=3000))['t_nw'] for s in range(20)])
print(f"null (edge=0), 20 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/20")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.004, seed=827, n_assets=9, n_days=4000))
print(f"planted (edge=0.004): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 20 seeds: NW t mean +0.01 (sd 1.22), |t|>=2 in 2/20


planted (edge=0.004): NW t = +2.52, Welch t = +2.87


## Verdict

- **Signal — None.** The single-name realized-skewness reversal does **not** carry up to the asset-class level. The long-low-skew / short-high-skew spread across nine class ETFs is **+13.73 bps/month** (NW *t* = **+0.62**) — right-signed but statistically zero, ~0.8σ into the placebo right tail (p = 0.20), flipping sign across the two eras (*t* = -0.45 / +1.39) and insignificant at every lookback. The 20-seed synthetic control fires on a *planted* relation (*t* = +2.52) and stays silent on the null (2/20), so the flat real result is a genuine absence of edge, not machinery.
- **Tradability — Mirage.** The gross spread is already insignificant (Sharpe 0.15); a token 5 bps one-way cost erases it to zero (net **+7.57 bps/mo** at 1 bp, -0.43 at 5 bps). Nothing survives friction because there was nothing there.